# Laboratorio: obtener y limpiar una noticia con BeautifulSoup

Este notebook muestra un flujo mínimo para:

1. Obtener una página web que contiene una noticia.
2. Leer su HTML.
3. Extraer el título y los párrafos principales.
4. Limpiar el texto.
5. Dejar la noticia en una estructura simple para procesamiento posterior.

> **Importante:** no todos los sitios permiten scraping automático. Antes de automatizar la descarga de noticias, se deben revisar los términos de uso del sitio y su archivo `robots.txt`.

## 1. Instalar e importar las librerías

Usaremos:

- `requests`: para descargar la página web.
- `BeautifulSoup`: para interpretar y recorrer el HTML.
- `re`: para hacer una limpieza básica del texto.

In [11]:
!pip -q install beautifulsoup4 requests

In [12]:
import requests
from bs4 import BeautifulSoup
import re

## 2. Indicar la URL de una noticia

Cambie la siguiente URL por una noticia que quiera analizar.

El código usa un `User-Agent` para que la petición se parezca a la realizada por un navegador convencional.

In [13]:
import requests

# URL de una noticia real publicada en Cooperativa
url = "https://www.cooperativa.cl/noticias/site/artic/20260902/pags-amp/20260902074532.html"

# Los headers contienen información adicional que se envía
# al servidor junto con la petición HTTP.
#
# User-Agent identifica el tipo de cliente que está haciendo
# la solicitud. En este caso simulamos un navegador Chrome.
#
# Algunos sitios web rechazan o limitan solicitudes realizadas
# por programas si no reciben un User-Agent reconocible.
headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

# Realizamos una petición HTTP GET al servidor.
# timeout=15 indica que esperaremos como máximo 15 segundos.
response = requests.get(
    url,
    headers=headers,
    timeout=15
)

print("Código HTTP:", response.status_code)
print("Cantidad de caracteres descargados:", len(response.text))

Código HTTP: 200
Cantidad de caracteres descargados: 56462


## 3. Interpretar el HTML con BeautifulSoup

Una página web contiene muchas etiquetas HTML:

```html
<h1>Título de la noticia</h1>
<p>Primer párrafo...</p>
<p>Segundo párrafo...</p>
```

BeautifulSoup transforma el HTML en una estructura que Python puede recorrer.

In [14]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.prettify()[:3000])

<!DOCTYPE html>
<html lang="es" ⚡="">
 <head>
  <!--MACRO METATAGS AMP-->
  <meta charset="utf-8"/>
  <meta content="width=device-width, user-scalable=yes" name="viewport"/>
  <!--SET SSI VARIABLES DESDE PRONTUS-->
  <!--solo articulos noticias -->
  <!--/solo articulos noticias -->
  <!-- REGIONES -->
  <!--/ REGIONES -->
  <!--SET SSI VARIABLES DESDE PRONTUS-->
  <!--METATAGS ESTANDAR-->
  <title>
   Falso runner que asaltaba a adolescentes fue detenido en La Reina - Cooperativa.cl
  </title>
  <meta content="index,follow" name="robots"/>
  <meta content="Carabineros indicó que el hombre de 42 años intimidaba con armas blancas a sus víctimas para quitarles sus celulares." name="description"/>
  <meta content="Falso runner que asaltaba a adolescentes fue detenido en La Reina" itemprop="name"/>
  <meta content="Carabineros indicó que el hombre de 42 años intimidaba con armas blancas a sus víctimas para quitarles sus celulares." itemprop="description"/>
  <meta content="  País,  Policia

## 4. Extraer el título

En muchos sitios el título principal está dentro de una etiqueta `<h1>`.

Este ejemplo busca el primer `<h1>` disponible.

In [15]:
h1 = soup.find("h1")

if h1:
    titulo = h1.get_text(" ", strip=True)
else:
    titulo = "Título no encontrado"

print(titulo)

Falso runner que asaltaba a adolescentes fue detenido en La Reina


## 5. Extraer los párrafos

La estrategia más simple consiste en recuperar todas las etiquetas `<p>`.

Luego descartamos párrafos demasiado cortos, porque suelen corresponder a menús, firmas, botones o elementos auxiliares.

In [16]:
parrafos = []

for p in soup.find_all("p"):
    texto = p.get_text(" ", strip=True)

    if len(texto) >= 40:
        parrafos.append(texto)

print("Párrafos encontrados:", len(parrafos))

for i, p in enumerate(parrafos[:5], start=1):
    print(f"\nPárrafo {i}:")
    print(p)

Párrafos encontrados: 7

Párrafo 1:
Carabineros indicó que el hombre de 42 años intimidaba con armas blancas a sus víctimas para quitarles sus celulares.

Párrafo 2:
El sujeto es sospechoso de cometer tres robos "haciéndose pasar por corredor", según Carabineros.

Párrafo 3:
Carabineros detuvo en la comuna de La Reina a un hombre de 42 años que se hacía pasar por "runner" para recorrer las calles, y así seleccionar a víctimas y robar sus teléfonos celulares.

Párrafo 4:
Según la mayor Carolina Constanzo , de la 16° Comisaría de La Reina, el sujeto es sospechoso de tres asaltos contra adolescentes , a quienes intimidaba con armas blancas .

Párrafo 5:
" Vistiendo ropa deportiva y haciéndose pasar por corredor , seguía a sus víctimas para, posteriormente, y de manera violenta, intimidarlas y sustraer sus especies, sobre todo sus teléfonos celulares ", precisó la oficial.


## 6. Construir el texto completo de la noticia

In [17]:
texto_noticia = "\n".join(parrafos)

print(texto_noticia[:4000])

Carabineros indicó que el hombre de 42 años intimidaba con armas blancas a sus víctimas para quitarles sus celulares.
El sujeto es sospechoso de cometer tres robos "haciéndose pasar por corredor", según Carabineros.
Carabineros detuvo en la comuna de La Reina a un hombre de 42 años que se hacía pasar por "runner" para recorrer las calles, y así seleccionar a víctimas y robar sus teléfonos celulares.
Según la mayor Carolina Constanzo , de la 16° Comisaría de La Reina, el sujeto es sospechoso de tres asaltos contra adolescentes , a quienes intimidaba con armas blancas .
" Vistiendo ropa deportiva y haciéndose pasar por corredor , seguía a sus víctimas para, posteriormente, y de manera violenta, intimidarlas y sustraer sus especies, sobre todo sus teléfonos celulares ", precisó la oficial.
Constanzo agregó que, por instrucción de la Fiscalía, el hombre pasó a un segundo control de detención por el delito de robo con violencia .
El sujeto además registra detenciones anteriores por los deli

## 7. Limpieza básica del texto

En esta etapa eliminaremos:

- espacios repetidos;
- saltos de línea excesivos;
- tabulaciones;
- espacios antes de signos de puntuación.

No queremos eliminar palabras ni modificar el contenido semántico de la noticia.

In [18]:
def limpiar_texto(texto):
    texto = texto.replace("\t", " ")
    texto = re.sub(r"[ ]+", " ", texto)
    texto = re.sub(r"\n\s*\n+", "\n", texto)
    texto = re.sub(r"\s+([,.;:!?])", r"\1", texto)
    return texto.strip()


texto_limpio = limpiar_texto(texto_noticia)

print(texto_limpio[:4000])

Carabineros indicó que el hombre de 42 años intimidaba con armas blancas a sus víctimas para quitarles sus celulares.
El sujeto es sospechoso de cometer tres robos "haciéndose pasar por corredor", según Carabineros.
Carabineros detuvo en la comuna de La Reina a un hombre de 42 años que se hacía pasar por "runner" para recorrer las calles, y así seleccionar a víctimas y robar sus teléfonos celulares.
Según la mayor Carolina Constanzo, de la 16° Comisaría de La Reina, el sujeto es sospechoso de tres asaltos contra adolescentes, a quienes intimidaba con armas blancas.
" Vistiendo ropa deportiva y haciéndose pasar por corredor, seguía a sus víctimas para, posteriormente, y de manera violenta, intimidarlas y sustraer sus especies, sobre todo sus teléfonos celulares ", precisó la oficial.
Constanzo agregó que, por instrucción de la Fiscalía, el hombre pasó a un segundo control de detención por el delito de robo con violencia.
El sujeto además registra detenciones anteriores por los delitos d

## 8. Guardar la noticia en una estructura Python

Por ahora no necesitamos una base de datos.

Podemos dejar la noticia en un diccionario. Más adelante este objeto puede enviarse a Gemini para extraer entidades y relaciones.

In [19]:
noticia = {
    "url": url,
    "titulo": titulo,
    "texto": texto_limpio
}

noticia

{'url': 'https://www.cooperativa.cl/noticias/site/artic/20260902/pags-amp/20260902074532.html',
 'titulo': 'Falso runner que asaltaba a adolescentes fue detenido en La Reina',
 'texto': 'Carabineros indicó que el hombre de 42 años intimidaba con armas blancas a sus víctimas para quitarles sus celulares.\nEl sujeto es sospechoso de cometer tres robos "haciéndose pasar por corredor", según Carabineros.\nCarabineros detuvo en la comuna de La Reina a un hombre de 42 años que se hacía pasar por "runner" para recorrer las calles, y así seleccionar a víctimas y robar sus teléfonos celulares.\nSegún la mayor Carolina Constanzo, de la 16° Comisaría de La Reina, el sujeto es sospechoso de tres asaltos contra adolescentes, a quienes intimidaba con armas blancas.\n" Vistiendo ropa deportiva y haciéndose pasar por corredor, seguía a sus víctimas para, posteriormente, y de manera violenta, intimidarlas y sustraer sus especies, sobre todo sus teléfonos celulares ", precisó la oficial.\nConstanzo agre

## 9. Guardar temporalmente como JSON

Este JSON puede utilizarse como entrada intermedia antes de generar la estructura Markdown para Obsidian.

In [20]:
import json

with open("noticia_limpia.json", "w", encoding="utf-8") as f:
    json.dump(noticia, f, ensure_ascii=False, indent=2)

print("Archivo creado: noticia_limpia.json")

Archivo creado: noticia_limpia.json


## 10. Mejora: buscar primero la etiqueta `<article>`

En muchos medios el contenido principal se encuentra dentro de una etiqueta `<article>`.
Esto ayuda a evitar menús, textos recomendados y otros elementos de la página.

In [21]:
article = soup.find("article")

if article:
    parrafos_article = [
        p.get_text(" ", strip=True)
        for p in article.find_all("p")
        if len(p.get_text(" ", strip=True)) >= 40
    ]

    texto_article = limpiar_texto("\n".join(parrafos_article))
    print(texto_article[:4000])
else:
    print("La página no contiene una etiqueta <article> reconocible.")

Carabineros indicó que el hombre de 42 años intimidaba con armas blancas a sus víctimas para quitarles sus celulares.
El sujeto es sospechoso de cometer tres robos "haciéndose pasar por corredor", según Carabineros.
Carabineros detuvo en la comuna de La Reina a un hombre de 42 años que se hacía pasar por "runner" para recorrer las calles, y así seleccionar a víctimas y robar sus teléfonos celulares.
Según la mayor Carolina Constanzo, de la 16° Comisaría de La Reina, el sujeto es sospechoso de tres asaltos contra adolescentes, a quienes intimidaba con armas blancas.
" Vistiendo ropa deportiva y haciéndose pasar por corredor, seguía a sus víctimas para, posteriormente, y de manera violenta, intimidarlas y sustraer sus especies, sobre todo sus teléfonos celulares ", precisó la oficial.
Constanzo agregó que, por instrucción de la Fiscalía, el hombre pasó a un segundo control de detención por el delito de robo con violencia.
El sujeto además registra detenciones anteriores por los delitos d

## 11. ¿Qué problemas pueden aparecer?

Este método es deliberadamente simple. En sitios reales puede fallar porque:

- el contenido se genera dinámicamente con JavaScript;
- el sitio bloquea peticiones automatizadas;
- el texto de la noticia está dentro de una estructura HTML específica;
- existen párrafos de publicidad o contenido recomendado;
- el título no está dentro de un `<h1>`;
- distintos medios tienen distintas estructuras HTML.

Por eso, en un laboratorio real conviene comenzar con uno o dos medios conocidos y adaptar los selectores HTML.

## Flujo obtenido

```text
URL
 ↓
requests
 ↓
HTML
 ↓
BeautifulSoup
 ↓
Título + párrafos
 ↓
Limpieza
 ↓
Texto estructurado
 ↓
JSON
 ↓
LLM / Gemini
```

En la siguiente etapa del laboratorio, el campo `texto` puede enviarse a Gemini para extraer información como:

- delito;
- personas;
- organizaciones;
- lugares;
- fechas;
- sustancias;
- relaciones entre entidades.